In [1]:
!cd /Users/hc284/Documents/_0taihuXomicsMicrobiome
!pwd

/Users/hc284/Documents/_0taihuXomicsMicrobiome


In [ ]:
"""extract mG<->mT
from the mG, mM, and mT elements from the pairwise correlation file"""
import pandas as pd
import re
from collections import defaultdict

mgt = pd.read_csv('pairwise_correlations_top_variables_top100.csv')
print(mgt.columns)
# pairwise correlation export
# first pair: mG - mM
mg = mgt.loc[(mgt['block1']=='mG') & (mgt['block2']=='mM'), mgt.columns[0]].unique()
mt = mgt.loc[(mgt['block1']=='mG') & (mgt['block2']=='mT'), mgt.columns[1]].unique()
mt = pd.Series(str).str.replace('mT_', 'UniRef90_')

ko = pd.read_table('summary_table_6_ko-full.tsv', sep='\t', low_memory=False)

mg_kopath = ko.loc[(ko['qseqid'].isin(mg)) & (ko['ko_pathway'] != 'NaN')]
mt_kopath = ko.loc[(ko['qseqid'].isin(mt)) & (ko['ko_pathway'] != 'NaN')]
# to summarize the pathway and metabolites of each mG gene
def enzy2pathmet(ko_df):
    dictEp = defaultdict(list)
    dictNEp = defaultdict(list)
    dictm = defaultdict(list)
    for i in range(ko_df.shape[0]):
        kk = ko_df.iloc[i]['ko_pathway']
        if kk != 'NaN':
            kv = ko_df.iloc[i]['qseqid']
            dictEp[kk].append(kv)
            if ko_df.iloc[i]['ko_reaction'] != 'NaN':
                mets = []
                for coln in ko_df.columns[11:]:
                    val_1 = ko_df.iloc[i][coln]
                    if not pd.isna(val_1) and val_1 != 0:
                        sides = re.split(r'\s*<=>\s*|\s*<=\s*|\s*=>\s*', val_1)
                        for side in sides:
                            for met in side.split('+'):
                                mets.append(met.strip())
                # Remove duplicates
                dictm[kk].extend(list(set(mets)))
        else:
            dictNEp[kk].append(ko_df.iloc[i]['qseqid'])
    return dictEp, dictNEp, dictm

mgdictEp, mgdictNEp, mgdictm = enzy2pathmet(mg_kopath)
with open('mG_mT_corr_mG-enzyPathway.tsv', 'w') as f:
    for key, value in dictEp.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")
with open('mG_mT_corr_mG-nonenzyPathway.tsv', 'w') as f:
    for key, value in dictNEp.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")
    
mtdictEp, mtdictNEp, mtdictm = enzy2pathmet(mt_kopath)
with open('mG_mT_corr_mt-enzyPathway.tsv', 'w') as f:
    for key, value in dictEp.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")
with open('mG_mT_corr_mt-nonenzyPathway.tsv', 'w') as f:
    for key, value in dictNEp.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")
     



Index(['var1', 'var2', 'correlation', 'block1', 'block2'], dtype='object')


In [ ]:
"""extract mG<->mM
from the mG, mM, and mT elements from the pairwise correlation file"""
import pandas as pd
import re
from collections import defaultdict

mgt = pd.read_csv('pairwise_correlations_top_variables_top100.csv')
print(mgt.columns)
# pairwise correlation export
# first pair: mG - mM
mg = mgt.loc[(mgt['block1']=='mG') & (mgt['block2']=='mM'), mgt.columns[0]].unique()
mm = mgt.loc[(mgt['block1']=='mG') & (mgt['block2']=='mM'), mgt.columns[1]].unique()

ko = pd.read_table('summary_table_6_ko-full.tsv', sep='\t', low_memory=False)
mets = pd.read_table('all_metabolome.txt', sep='\t', low_memory=False)

mg_kopath = ko.loc[(ko['qseqid'].isin(mg)) & (ko['ko_pathway'] != 'NaN')]
mm_mets = mets.loc[mets['ID'].isin(mm)]
def mM2Mets(ko_df):
    dictM = defaultdict(list)
    
# to summarize the pathway and metabolites of each mG gene
def enzy2pathmet(ko_df):
    dictEp = defaultdict(list)
    dictNEp = defaultdict(list)
    dictm = defaultdict(list)
    for i in range(ko_df.shape[0]):
        kk = ko_df.iloc[i]['ko_pathway']
        if kk != 'NaN':
            kv = ko_df.iloc[i]['qseqid']
            dictEp[kk].append(kv)
            if ko_df.iloc[i]['ko_reaction'] != 'NaN':
                mets = []
                for coln in ko_df.columns[11:]:
                    val_1 = ko_df.iloc[i][coln]
                    if not pd.isna(val_1) and val_1 != 0:
                        sides = re.split(r'\s*<=>\s*|\s*<=\s*|\s*=>\s*', val_1)
                        for side in sides:
                            for met in side.split('+'):
                                mets.append(met.strip())
                # Remove duplicates
                dictm[kk].extend(list(set(mets)))
        else:
            dictNEp[kk].append(ko_df.iloc[i]['qseqid'])
    return dictEp, dictNEp, dictm

dictEp, dictNEp, dictm = enzy2pathmet(mg_kopath)

with open('mG_mT_corr_mG-enzyPathway.tsv', 'w') as f:
    for key, value in dictEp.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")
with open('mG_mT_corr_mG-nonenzyPathway.tsv', 'w') as f:
    for key, value in dictNEp.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")
with open('mG_mT_corr_mM-list.tsv', 'w') as f:
    for key, value in dictm.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")        


In [ ]:
"""extract mM<->mT from tri_pairwise corr b/w mG, mM, and mT"""
import pandas as pd
import re
from collections import defaultdict

mgt = pd.read_csv('pairwise_correlations_top_variables_top100.csv')
print(mgt.columns)
# pairwise correlation export
# first pair: mM - mT
mm = mgt.loc[(mgt['block1']=='mM') & (mgt['block2']=='mT'), mgt.columns[0]].unique()
mt = mgt.loc[(mgt['block1']=='mM') & (mgt['block2']=='mT'), mgt.columns[1]].unique()
mt = pd.Series(str).str.replace('mT_', 'UniRef90_')

ko = pd.read_table('summary_table_6_ko-full.tsv', sep='\t', low_memory=False)

mg_kopath = ko.loc[(ko['qseqid'].isin(mg)) & (ko['ko_pathway'] != 'NaN')]
mt_kopath = ko.loc[(ko['qseqid'].isin(mt)) & (ko['ko_pathway'] != 'NaN')]
# to summarize the pathway and metabolites of each mG gene
def enzy2pathmet(ko_df):
    dictEp = defaultdict(list)
    dictNEp = defaultdict(list)
    dictm = defaultdict(list)
    for i in range(ko_df.shape[0]):
        kk = ko_df.iloc[i]['ko_pathway']
        if kk != 'NaN':
            kv = ko_df.iloc[i]['qseqid']
            dictEp[kk].append(kv)
            if ko_df.iloc[i]['ko_reaction'] != 'NaN':
                mets = []
                for coln in ko_df.columns[11:]:
                    val_1 = ko_df.iloc[i][coln]
                    if not pd.isna(val_1) and val_1 != 0:
                        sides = re.split(r'\s*<=>\s*|\s*<=\s*|\s*=>\s*', val_1)
                        for side in sides:
                            for met in side.split('+'):
                                mets.append(met.strip())
                # Remove duplicates
                dictm[kk].extend(list(set(mets)))
        else:
            dictNEp[kk].append(ko_df.iloc[i]['qseqid'])
    return dictEp, dictNEp, dictm

mgdictEp, mgdictNEp, mgdictm = enzy2pathmet(mg_kopath)
with open('mG_mT_corr_mG-enzyPathway.tsv', 'w') as f:
    for key, value in dictEp.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")
with open('mG_mT_corr_mG-nonenzyPathway.tsv', 'w') as f:
    for key, value in dictNEp.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")
    
mtdictEp, mtdictNEp, mtdictm = enzy2pathmet(mt_kopath)
with open('mG_mT_corr_mt-enzyPathway.tsv', 'w') as f:
    for key, value in dictEp.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")
with open('mG_mT_corr_mt-nonenzyPathway.tsv', 'w') as f:
    for key, value in dictNEp.items():
        values_str = ','.join(value)
        f.write(f"{key}\t{values_str}\n")
     

